# LRU的实现方式

- 有序字典
- hash表 + 双向队列

## 有序字典

主要使用

有序字典
```
from collections import OrderedDict
```
**popitem(last=True)** 
- last=True  按照 LIFO 的顺序删除
- last=False 按照 FIFO 的顺序删除

**move_to_end(key, last=True)**
- last=True  按照 LIFO 的顺序将key对移动到末尾
- last=False 按照 FIFO 的顺序将key对移动到末尾


In [69]:
from collections import OrderedDict
class LRUCache(OrderedDict):

    def __init__(self, capacity):
        """
        :type capacity: int
        """
        self.capacity = capacity

    def get(self, key):
        """
        :type key: int
        :rtype: int
        """
        if key not in self:
            return - 1
        
        self.move_to_end(key)
        return self[key]

    def put(self, key, value):
        """
        :type key: int
        :type value: int
        :rtype: void
        """
        if key in self:
            self.move_to_end(key)
        self[key] = value
        if len(self) > self.capacity:
            self.popitem(last = False)



## 双向队列和hash表

因为同时使用了双向队列和hash表来维护LRU,
所以分别需要两套对应的删除和添加机制
- 双向链表的添加和删除
- hash表的添加和删除

LRU中添加操作和删除操作仅仅在put方法中出现

> Tips:  
> 使用等号进行交换时  
> 引用类型最好不要直接交换  
> 数值问题随意交换

In [65]:
class DLinkNode:
    def __init__(self):
        self.key = 0
        self.value = 0
        self.prev = None
        self.next = None

class LRUCache:
    
    def _add_node(self,node):
        # 添加到头部,最近最先使用
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node
        
    def _remove_node(self,node):
        node.prev.next,node.next.prev = node.next,node.prev
    
    def _move_to_head(self,node):
        self._remove_node(node)
        self._add_node(node)
    
    def _pop_tail(self):
        tail = self.tail.prev
        self._remove_node(tail)
        return tail
    
    def __init__(self,capacity):
        self.cache = {} # hash
        self.size = 0
        self.capacity = capacity
        
        # 这一部分就是双向链表
        self.head,self.tail = DLinkNode(),DLinkNode()
        self.head.next = self.tail
        self.tail.prev = self.head
    
    def get(self,key):
        
        if key not in self.cache:
            return -1

        node = self.cache[key]
        
        self._move_to_head(node)
        return node.value


    def put(self,key,value):
        # 如果包含key就不需要处理cache
        # 仅仅处理队列就可以
        # 如果cache不包含key就需要同时处理 cache和双向队列
        
        if key in self.cache:
            node = self.cache[key]
            node.value = value
            self._move_to_head(node)
        else:
            new_node = DLinkNode()
            new_node.key = key
            new_node.value = value
            # 同时添加到 cache 和 双向队列中
            self.cache[key] =  new_node
            self._add_node(new_node)
            # 判断是否需要处理删除
            self.size +=1
            
            if self.size > self.capacity:
                # 双向队列的删除
                # cache的删除
                tail = self._pop_tail()     
                print("del key : ",tail.value)
                del self.cache[tail.key]
                self.size -= 1

In [68]:
cache = LRUCache(2)
cache.put(1,1)
cache.put(2, 2)
cache.get(1)
cache.put(3, 3)
cache.get(2)

del key :  2


-1

In [46]:
cache.head.next.next.next.value

1